In [5]:
import sys, os
import numpy as np
import torch
import json
from tqdm import tqdm
from omegaconf import OmegaConf
import secrets
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import copy
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# insert path to your cleo/reinforced_mpnn repo here
sys.path.append("/home/jgershon/git/cleo/reinforced_mpnn")
from policy_utils import PolicyMPNN
from fragment_utils import make_fragment_dict, sample_sequences


### Sample sequences and generate fragment dictionary

In [ ]:
# path to save fragment dictionary
save_path = "/home/jgershon/projects/itopt/kiera/policy_sampled_fragments/sampled_fragment_dictionary.json"

# base path where policy checkpoints are stored
base_path = "/home/jgershon/projects/itopt/policy_mpnn"

# runs tags and steps to sample from
run_names = ["petase_v16_refw5","petase_v16_refw8"]
steps = [150, 200, 250]

# number of batches to sample from each checkpoint (will sample with batch size found in config (usually 32))
num_batches_to_sample = 16

# fragment bounds to use
fragment_bounds = [[0, 41], [42, 85], [86, 127], [128, 164], [165, 208]]


# sampling loop
fragment_dict = {f"fragment_{i+1}":[] for i in range(len(fragment_bounds))}
for run in run_names:
    for step in steps:
        checkpoint_path = os.path.join(base_path, run, f"{run}_step_{step:04}.pt")
        ckpt = torch.load(checkpoint_path, map_location="cpu")

        # set checkpoint path to load 
        cfg = OmegaConf.create(ckpt["config"])
        cfg.checkpoint_path = checkpoint_path

        # sample
        policy = PolicyMPNN(cfg)
        sequences = policy.sample_from_policy(num_batches_to_sample)

        # get frags
        frag_dict = make_fragment_dict(sequences, fragment_bounds)

        # add unique fragments to frag dict
        for f in frag_dict:
            unique_frags = list(set([x[1] for x in frag_dict[f]]))
            to_add = [(f"{run}.step{step:04}.{n:04}.{secrets.token_hex(4)}", frag) for n, frag in enumerate(unique_frags)]

            fragment_dict[f].extend(to_add)


assert not os.path.exists(save_path), f"Error: {save_path} already exists, please delete before running script"

with open(save_path, "w") as f:
    json.dump(fragment_dict, f, indent=4)

Loading training checkpoint from /home/jgershon/projects/itopt/policy_mpnn/petase_v16_refw5/petase_v16_refw5_step_0150.pt


100%|██████████| 16/16 [01:34<00:00,  5.89s/it]


Loading training checkpoint from /home/jgershon/projects/itopt/policy_mpnn/petase_v16_refw5/petase_v16_refw5_step_0200.pt


100%|██████████| 16/16 [01:34<00:00,  5.90s/it]


Loading training checkpoint from /home/jgershon/projects/itopt/policy_mpnn/petase_v16_refw5/petase_v16_refw5_step_0250.pt


...

100%|██████████| 16/16 [01:34<00:00,  5.90s/it]


Loading training checkpoint from /home/jgershon/projects/itopt/policy_mpnn/petase_v16_refw8/petase_v16_refw8_step_0200.pt


100%|██████████| 16/16 [01:34<00:00,  5.90s/it]


Loading training checkpoint from /home/jgershon/projects/itopt/policy_mpnn/petase_v16_refw8/petase_v16_refw8_step_0250.pt


100%|██████████| 16/16 [01:34<00:00,  5.90s/it]


### Load fragment dictionary and sample seqeunces from library to fold

In [ ]:
# insert path to fragment dictionary here (saved above)
fragment_dict_path = "/home/jgershon/projects/itopt/kiera/policy_sampled_fragments/sampled_fragment_dictionary.json"

with open(fragment_dict_path, "r") as f:
    fragment_dict = json.load(f)

# tupleize the fragments
for k in fragment_dict:
    fragment_dict[k] = [(x[0], x[1]) for x in fragment_dict[k]]

print("Fragment counts:")
for k in fragment_dict:
    print(f"{k} : {len(fragment_dict[k])}")

fragment_1 : 2875
fragment_2 : 2149
fragment_3 : 2850
fragment_4 : 1953
fragment_5 : 1974


In [ ]:
# number of sequences to sample
num_samples = 15000

# min number of samples per fragment
min_sample = 2

# path to save fasta file
fasta_path = "/home/jgershon/projects/itopt/kiera/policy_sampled_fragments/sampled_sequences_to_fold.fasta"


# sample sequences
samples = sample_sequences(fragment_dict, num_samples, min_sample, max_iter=5, join_char="___")


# write to fasta file
fasta_lines = []
for name, seq in samples:
    fasta_lines.append(f">{name}\n{seq}\n")

assert not os.path.exists(fasta_path), "Fasta file already exists! Will not overwrite."

with open(fasta_path, "w") as f:
    f.writelines(fasta_lines)